In [8]:
import pickle
import networkx as nx
import pennylane as qml
from joblib import Parallel, delayed
from pennylane import numpy as np
from itertools import combinations

In [68]:
def aon_hamiltonian(edge):
    edge_length = len(edge)

    minus_terms = []
    plus_terms = []

    for i in edge:
        minus_terms.append(
            qml.Identity(i) - qml.PauliZ(i)
        )

        plus_terms.append(
            qml.Identity(i) + qml.PauliZ(i)
        )

    minus_product = minus_terms[0]
    plus_product = plus_terms[0]

    for op in minus_terms[1:]:
        minus_product = minus_product @ op

    for op in plus_terms[1:]:
        plus_product = plus_product @ op

    return (qml.Identity(edge[0])- (1 / (2 ** edge_length))* (minus_product + plus_product))

In [69]:
def aon_cost_hamiltonian(edges):
    H = 0

    for edge in edges:
        H = H + aon_hamiltonian(edge)

    return H

In [70]:
dev = qml.device("default.qubit",wires=6)

In [71]:
@qml.qnode(dev)
def basis_cost(x):
    qml.BasisState(np.array(x, dtype=int),wires=range(6))
    return qml.expval(H_C)

In [72]:
x = [0,0,0,1,1,1]
x2 = [1,1,1,0,0,0]
x3=[1,0,1,0,1,0]
test=[x,x2,x3]
edges = [(0, 1, 2),(3, 4, 5),(0, 3)]
for bitstring in test:
    H_C = aon_cost_hamiltonian(edges)
    print(basis_cost(bitstring))



1.0
1.0
3.0


In [66]:
def dicke_state(n, weight):
    state = np.zeros(2**n, dtype=complex)
    states = list(combinations(range(n), weight))
    amplitude = 1 / np.sqrt(len(states))
    for ones in states:
        bits = np.zeros(n, dtype=int)
        bits[list(ones)] = 1
        index = int("".join(map(str, bits)), 2)
        state[index] = amplitude
    return state

In [67]:
psi0 = dicke_state(6, 3)

print(np.sum(np.abs(psi0)**2))
print(np.count_nonzero(psi0))

1.0
20


In [19]:
@qml.qnode(dev)
def dicke_cost():
    qml.StatePrep(
        psi0,
        wires=range(6)
    )

    return qml.expval(H_C)

In [20]:
print(dicke_cost())

2.4


In [46]:
import itertools
def clique_expansion(hyperedges):
    mixer_edges = set()

    for edge in hyperedges:
        for u, v in itertools.combinations(edge, 2):
            mixer_edges.add(tuple(sorted((u, v))))

    return sorted(mixer_edges)

In [47]:
mixer_edges = clique_expansion(edges)
print(mixer_edges)


[(0, 1), (0, 2), (0, 3), (1, 2), (3, 4), (3, 5), (4, 5)]


In [76]:
def swap_mixer(beta,edge_list):
    for u,v in edge_list: 
        qml.S(wires=u)
        qml.SingleExcitationMinus(-2*beta,wires=[u,v])
        qml.adjoint(qml.S)(wires=u)
        

In [80]:
@qml.qnode(dev)
def mixer_test(beta):
    qml.StatePrep(psi0,wires=range(6))
    swap_mixer(beta,mixer_edges)
    return qml.probs(wires=range(6))

In [81]:
probs = mixer_test(np.pi/9)

In [82]:
total=0
for i, p in enumerate(probs):
    if p > 1e-10:
        total+=1
        bitstring = format(i, "06b")

        print(
            bitstring,
            "weight =", bitstring.count("1"),
            "prob =", p
        )
print(f"total states: {total}")

000111 weight = 3 prob = 0.05000000000000002
001011 weight = 3 prob = 0.05000000000000002
001101 weight = 3 prob = 0.05000000000000002
001110 weight = 3 prob = 0.05000000000000002
010011 weight = 3 prob = 0.05000000000000002
010101 weight = 3 prob = 0.05000000000000002
010110 weight = 3 prob = 0.05000000000000002
011001 weight = 3 prob = 0.05000000000000002
011010 weight = 3 prob = 0.05000000000000002
011100 weight = 3 prob = 0.05000000000000002
100011 weight = 3 prob = 0.05000000000000002
100101 weight = 3 prob = 0.05000000000000002
100110 weight = 3 prob = 0.05000000000000002
101001 weight = 3 prob = 0.05000000000000002
101010 weight = 3 prob = 0.05000000000000002
101100 weight = 3 prob = 0.05000000000000002
110001 weight = 3 prob = 0.05000000000000002
110010 weight = 3 prob = 0.05000000000000002
110100 weight = 3 prob = 0.05000000000000002
111000 weight = 3 prob = 0.05000000000000002
total states: 20


In [85]:
@qml.qnode(dev)
def combined_circuit(beta):

    # Prepare balanced Dicke state
    qml.StatePrep(psi0,wires=range(6))

    # Apply SWAP mixer
    swap_mixer(beta,mixer_edges)

    # Measure the AoN cost Hamiltonian
    return qml.expval(H_C)

In [86]:
cost = combined_circuit(np.pi / 9)

print(cost)

2.4000000000000012


In [87]:
H_C = aon_cost_hamiltonian(edges)
def cost_unitary(gamma):
    qml.exp(H_C,coeff=-1j * gamma)

In [88]:
@qml.qnode(dev)
def qaoa_circuit(gamma, beta):

    qml.StatePrep(psi0,wires=range(6))
    cost_unitary(gamma)
    swap_mixer(beta,mixer_edges)

    return qml.expval(H_C)

In [89]:
gamma = np.pi / 4
beta = np.pi / 9

cost = qaoa_circuit(gamma, beta)

print("Expected cost:", cost)

Expected cost: 2.1060465903537393


In [91]:
print(qml.draw(qaoa_circuit)(gamma,beta))

0: ─╭|Ψ⟩─╭Exp(0.00-0.79j 𝓗)──S─╭G₋(-0.70)──S†──S─╭G₋(-0.70)──S†──S─╭G₋(-0.70)──S†─────────────── ···
1: ─├|Ψ⟩─├Exp(0.00-0.79j 𝓗)────╰G₋(-0.70)────────│─────────────────│───────────S──╭G₋(-0.70)──S† ···
2: ─├|Ψ⟩─├Exp(0.00-0.79j 𝓗)──────────────────────╰G₋(-0.70)────────│──────────────╰G₋(-0.70)──── ···
3: ─├|Ψ⟩─├Exp(0.00-0.79j 𝓗)────────────────────────────────────────╰G₋(-0.70)──S──╭G₋(-0.70)──S† ···
4: ─├|Ψ⟩─├Exp(0.00-0.79j 𝓗)───────────────────────────────────────────────────────╰G₋(-0.70)──── ···
5: ─╰|Ψ⟩─╰Exp(0.00-0.79j 𝓗)───────────────────────────────────────────────────────────────────── ···

0: ··· ──────────────────────────────────┤ ╭<𝓗>
1: ··· ──────────────────────────────────┤ ├<𝓗>
2: ··· ──────────────────────────────────┤ ├<𝓗>
3: ··· ──S─╭G₋(-0.70)──S†────────────────┤ ├<𝓗>
4: ··· ────│───────────S──╭G₋(-0.70)──S†─┤ ├<𝓗>
5: ··· ────╰G₋(-0.70)─────╰G₋(-0.70)─────┤ ╰<𝓗>


In [96]:
decomposed = qml.transforms.decompose(
    qaoa_circuit,
    gate_set={"RX","RY","RZ","CNOT","S","Adjoint(S)","SingleExcitationMinus","Exp"}
)

In [97]:
print(
    qml.draw(decomposed)(
        gamma,
        beta
    )
)

0: ──RY(1.57)─╭●───────────╭●──────────────╭●─────────────────────────╭●──────────────── ···
1: ──RY(1.57)─╰X──RY(0.20)─╰X─╭●───────────│─────────────╭●───────────│───────────────╭● ···
2: ──RY(1.57)─────────────────╰X──RY(0.26)─╰X──RY(-0.00)─╰X──RY(0.26)─╰X─╭●───────────│─ ···
3: ──RY(1.57)────────────────────────────────────────────────────────────╰X──RY(0.48)─╰X ···
4: ──RY(1.37)─────────────────────────────────────────────────────────────────────────── ···
5: ──RY(0.98)─────────────────────────────────────────────────────────────────────────── ···

0: ··· ─────────────────────────╭●──────────────────────────────────────────────────╭●─── ···
1: ··· ─────────────────────────│─────────────────────────╭●────────────────────────│──── ···
2: ··· ────────────╭●───────────│────────────╭●───────────│────────────╭●───────────│──── ···
3: ··· ──RY(-0.00)─╰X──RY(0.48)─╰X──RY(0.00)─╰X──RY(0.14)─╰X──RY(0.00)─╰X──RY(0.48)─╰X─╭● ···
4: ··· ──────────────────────────────────────────────────────────